## MovieLens 1M
Para este proyecto usaremos el DataSet de MovieLens 1M

In [1]:
import os
import kagglehub
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [2]:
#Download latest version
path = kagglehub.dataset_download("odedgolden/movielens-1m-dataset")

print("Path to dataset files:", path)

100%|██████████| 5.83M/5.83M [00:00<00:00, 8.33MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/odedgolden/movielens-1m-dataset/versions/1


In [3]:
print(os.listdir(path))

['ratings.dat', 'movies.dat', 'README', 'users.dat']


Cargamos todos los datos, ya que usaremos las interacciones y las informaciones de peliculas y usuarios

In [4]:
ratings = pd.read_csv(path + "/ratings.dat", sep="::", engine="python",
                      names=["user_id", "movie_id", "rating", "timestamp"])

movies = pd.read_csv(path + "/movies.dat", sep="::", engine="python",
                     names=["movie_id", "title", "genres"], encoding="latin-1" )

users = pd.read_csv(path + "/users.dat", sep="::", engine="python",
                    names=["user_id", "gender", "age", "occupation", "zip"])

In [5]:
# Limpiar títulos y extraer año
movies['year'] = movies['title'].str.extract(r'\((\d{4})\)')
movies['clean_title'] = movies['title'].str.replace(r'\s*\(\d{4}\)', '', regex=True).str.strip()

print(movies[['title', 'clean_title', 'year']].head())

                                title                  clean_title  year
0                    Toy Story (1995)                    Toy Story  1995
1                      Jumanji (1995)                      Jumanji  1995
2             Grumpier Old Men (1995)             Grumpier Old Men  1995
3            Waiting to Exhale (1995)            Waiting to Exhale  1995
4  Father of the Bride Part II (1995)  Father of the Bride Part II  1995


In [8]:
df = ratings.merge(movies, on="movie_id").merge(users, on="user_id")

Implementacion de al menos 3 modelos de referencia

In [9]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(df, test_size=0.2, random_state=42)

Consideramos relevantes de 3 hacia arriba

In [10]:
test_user_likes = test[test["rating"] >= 3].groupby("user_id")["movie_id"].apply(set)

In [11]:
def precision_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]
    relevant_set = set(relevant)

    if len(recommended_k) == 0:
        return 0

    hits = len(set(recommended_k) & relevant_set)
    return hits / k

In [28]:
def recall_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]
    relevant_set = set(relevant)

    if len(relevant_set) == 0:
        return 0

    hits = len(set(recommended_k) & relevant_set)
    return hits / len(relevant_set)

In [29]:
def dcg_at_k(recommended, relevant, k):
    recommended = recommended[:k]
    dcg = 0.0
    for i, item in enumerate(recommended):
        if item in relevant:
            dcg += 1 / np.log2(i + 2)
    return dcg

def ndcg_at_k(recommended, relevant, k):
    dcg = dcg_at_k(recommended, relevant, k)
    ideal = dcg_at_k(list(relevant)[:k], relevant, k)
    if ideal == 0:
        return 0.0
    return dcg / ideal

def evaluate_model(model_func, users, k=10):
    precisions = []
    recalls = []
    ndcgs = []

    for user in users:
        if user not in test_user_likes:
            continue

        relevant = test_user_likes[user]
        recommended = model_func(user, k)

        precisions.append(precision_at_k(recommended, relevant, k))
        recalls.append(recall_at_k(recommended, relevant, k))
        ndcgs.append(ndcg_at_k(recommended, relevant, k))

    return np.mean(precisions), np.mean(recalls), np.mean(ndcgs)

NDCG agregado con ia https://claude.ai/share/a7e98c09-b3c0-40f6-8ace-7172a75569b8

Random

In [30]:
all_movies = train["movie_id"].unique()

def random_recommender(user_id, k=10):
    return np.random.choice(all_movies, size=k, replace=False)

In [31]:
prec_r, rec_r, ndcg = evaluate_model(random_recommender, df["user_id"].unique())

In [32]:
print("Random -> Precision:", prec_r, "Recall:", rec_r, "NDCG:", ndcg)


Random -> Precision: 0.0071428571428571435 Recall: 0.002628073176573592 NDCG: 0.007253430151928839


Most *popular*

In [33]:
movie_popularity = train.groupby("movie_id").size().sort_values(ascending=False)
def most_popular_recommender(user_id, k=10):
    return movie_popularity.head(k).index.tolist()

In [34]:
prec_p, rec_p, ndcg = evaluate_model(most_popular_recommender, df["user_id"].unique())

In [35]:
print("Popular -> Precision:", prec_p, "Recall:", rec_p, "NDCG:", ndcg)


Popular -> Precision: 0.08882996353994034 Recall: 0.04893987738464019 NDCG: 0.09852164278786607


Corrección de bug asistida pro IA: https://chatgpt.com/share/69fa06c5-daa4-8333-8ea8-2db98013231e


Modelo 3:

In [36]:
#train, test = train_test_split(df, test_size=0.2, random_state=42)

In [37]:
!pip install pyreclab --upgrade

In [38]:
import pyreclab

In [39]:
train[["user_id", "movie_id", "rating"]].to_csv("data_train", sep="\t", index=False, header=False)

In [40]:
test[["user_id", "movie_id", "rating"]].to_csv("data_test", sep="\t", index=False, header=False)

In [41]:
# Definicion de objeto svd
svd = pyreclab.SVD(dataset="data_train",
                   dlmchar=b'\t',
                   header=False,
                   usercol=0,
                   itemcol=1,
                   ratingcol=2)

# Entrenamiento del modelo
svd.train(factors=80, maxiter=50, lr=0.01, lamb=0.1)

In [45]:
# Testing de recomendaciones
top_n = 10

recommendList, maprec, ndcg = svd.testrec(input_file='data_test',
                                          dlmchar=b'\t',
                                          header=False,
                                          usercol=0,
                                          itemcol=1,
                                          ratingcol=2,
                                          topn=top_n,
                                          relevance_threshold=3,
                                          includeRated=False)

print('MAP: {}\nNDCG@{}: {}'.format(maprec, top_n, ndcg))

MAP: 0.08503921802910541
NDCG@10: 0.0414996921439811


In [46]:
print(recommendList)

{'1': ['318', '3245', '787', '953', '593', '904', '2503', '53', '1262', '2905'], '10': ['3245', '2197', '53', '527', '1172', '3147', '2503', '214', '2342', '2905'], '100': ['3245', '53', '2503', '2905', '2999', '1148', '787', '1207', '2019', '904'], '1000': ['3245', '2905', '50', '53', '2762', '2503', '1172', '1148', '557', '745'], '1001': ['3245', '787', '750', '106', '2019', '1178', '922', '2905', '1076', '2731'], '1002': ['3245', '2019', '787', '2905', '53', '750', '50', '1148', '1172', '3338'], '1003': ['3245', '2905', '2762', '2503', '50', '2571', '53', '1262', '904', '1148'], '1004': ['3245', '2905', '318', '50', '557', '1148', '2503', '2762', '3338', '527'], '1005': ['3245', '318', '50', '2905', '1148', '2019', '858', '904', '745', '913'], '1006': ['3245', '858', '2905', '50', '296', '2858', '1221', '318', '1193', '2019'], '1007': ['1743', '787', '2503', '53', '3517', '2905', '318', '2762', '1198', '2937'], '1008': ['3245', '53', '2905', '787', '2019', '557', '2503', '50', '750'

In [47]:
k = 10
precisions = []
recalls = []
ndcgs = []

for user_str, recommended in recommendList.items():
    user = int(user_str)

    if user not in test_user_likes.index:
        continue

    relevant = test_user_likes[user]
    recommended_ids = [int(m) for m in recommended]

    precisions.append(precision_at_k(recommended_ids, relevant, k))
    recalls.append(recall_at_k(recommended_ids, relevant, k))
    ndcgs.append(ndcg_at_k(recommended_ids, relevant, k))

prec_svd = np.mean(precisions)
rec_svd = np.mean(recalls)
ndcg_svd = np.mean(ndcgs)

print(f"SVD -> Precision@{k}: {prec_svd:.4f} | Recall@{k}: {rec_svd:.4f}")
print(ndcg_svd)

SVD -> Precision@10: 0.0426 | Recall@10: 0.0198
0.04149969212591775


EL último bloque fue creado con IA, para cambiar el formato de recommendsList para que sea compatible con las funciones definidas anteriormente https://claude.ai/share/c5dd9f04-a378-41b2-a50d-d86fbd713b59
